In [1]:
import numpy as np

In [2]:
seed = 1
n_in = 2
n_hidden = 4
lr = 1.0

In [3]:
X = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])
y = np.array([0, 1, 1, 0])

## init

In [4]:
rng = np.random.default_rng(seed)

In [5]:
W1 = rng.normal(0, 1/np.sqrt(n_in),size=(n_in, n_hidden))
W1.shape

(2, 4)

In [6]:
W1

array([[ 0.24436493,  0.58097176,  0.2336543 , -0.92147132],
       [ 0.64018327,  0.31563449, -0.37968327,  0.41091255]])

In [7]:
b1 = np.zeros(n_hidden)
b1

array([0., 0., 0., 0.])

In [8]:
W2 = rng.normal(0, 1/np.sqrt(n_hidden), size=(n_hidden, 1))
W2.shape

(4, 1)

In [9]:
W2

array([[0.1822862 ],
       [0.14706625],
       [0.01421112],
       [0.27335649]])

In [10]:
b2 = np.zeros(1)
b2.shape

(1,)

In [11]:
b2

array([0.])

## forward

In [12]:
z1 = X @ W1 + b1
X.shape, W1.shape, b1.shape, z1.shape

((4, 2), (2, 4), (4,), (4, 4))

In [13]:
z1

array([[ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.64018327,  0.31563449, -0.37968327,  0.41091255],
       [ 0.24436493,  0.58097176,  0.2336543 , -0.92147132],
       [ 0.8845482 ,  0.89660625, -0.14602898, -0.51055876]])

In [14]:
def sigmoid(z):
    # 为数值稳定用 np.clip
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def dsigmoid(a):
    """sigmoid 对 a 本身求导（a 已经是 sigmoid(z)）"""
    return a * (1.0 - a)

In [15]:
a1 = sigmoid(z1)
a1.shape

(4, 4)

In [16]:
a1

array([[0.5       , 0.5       , 0.5       , 0.5       ],
       [0.65479489, 0.57825998, 0.40620329, 0.60130667],
       [0.56078903, 0.64129098, 0.55814926, 0.2846582 ],
       [0.70776383, 0.71025159, 0.46355749, 0.37506255]])

In [17]:
z2 = a1 @ W2 + b2
a1.shape, W2.shape, b2.shape, z2.shape

((4, 4), (4, 1), (1,), (4, 1))

In [18]:
z2

array([[0.30846003],
       [0.37454628],
       [0.28228145],
       [0.34258307]])

In [19]:
y_pred = sigmoid(z2).ravel()
y_pred.shape

(4,)

In [20]:
y_pred

array([0.57650933, 0.59255706, 0.57010546, 0.58481784])

In [21]:
# 交叉熵损失（加 1e-8 防止 log(0)）
loss = -np.mean(y * np.log(y_pred + 1e-8) +
                (1 - y) * np.log(1 - y_pred + 1e-8))

In [22]:
loss

np.float64(0.7058759120999232)

## backward

In [23]:
N = len(X)
N

4

In [24]:
y_pred = y_pred.reshape(-1, 1)
y_pred

array([[0.57650933],
       [0.59255706],
       [0.57010546],
       [0.58481784]])

In [25]:
dz2 = (y_pred - y.reshape(-1, 1))
dz2

array([[ 0.57650933],
       [-0.40744294],
       [-0.42989454],
       [ 0.58481784]])

In [26]:
dW2 = (a1.T @ dz2) / N
dW2

array([[0.04857397],
       [0.04808176],
       [0.03847534],
       [0.03505669]])

In [27]:
db2 = dz2.mean(axis=0) 
db2

array([0.08099742])

In [28]:
da1 = dz2 @ W2.T
da1

array([[ 0.10508969,  0.08478506,  0.00819284,  0.15759257],
       [-0.07427122, -0.0599211 , -0.00579022, -0.11137717],
       [-0.07836384, -0.06322298, -0.00610928, -0.11751446],
       [ 0.10660422,  0.08600697,  0.00831092,  0.15986376]])

In [29]:
dz1 = da1 * dsigmoid(a1)
dz1

array([[ 0.02627242,  0.02119627,  0.00204821,  0.03939814],
       [-0.01678816, -0.01461328, -0.00139661, -0.02670122],
       [-0.01930138, -0.01454362, -0.00150666, -0.02392922],
       [ 0.0220494 ,  0.01769974,  0.00206669,  0.03747057]])

In [30]:
dW1 = (X.T @ dz1) / N
dW1

array([[0.000687  , 0.00078903, 0.00014001, 0.00338534],
       [0.00131531, 0.00077161, 0.00016752, 0.00269234]])

In [31]:
db1 = dz1.mean(axis=0)
db1

array([0.00305807, 0.00243478, 0.00030291, 0.00655956])

## Gradient Descent

In [32]:
W1 -= lr * dW1
b1 -= lr * db1
W2 -= lr * dW2
b2 -= lr * db2
W1, b1, W2, b2

(array([[ 0.24367792,  0.58018273,  0.23351429, -0.92485665],
        [ 0.63886796,  0.31486287, -0.37985079,  0.40822022]]),
 array([-0.00305807, -0.00243478, -0.00030291, -0.00655956]),
 array([[ 0.13371223],
        [ 0.09898449],
        [-0.02426422],
        [ 0.2382998 ]]),
 array([-0.08099742]))

In [33]:
z1 = X @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2
a2 = sigmoid(z2).ravel()
z1, a1, z2, a2

(array([[-3.05807005e-03, -2.43477750e-03, -3.02906421e-04,
         -6.55956497e-03],
        [ 6.35809893e-01,  3.12428095e-01, -3.80153700e-01,
          4.01660652e-01],
        [ 2.40619852e-01,  5.77747952e-01,  2.33211384e-01,
         -9.31416216e-01],
        [ 8.79487815e-01,  8.92610824e-01, -1.46639410e-01,
         -5.23195999e-01]]),
 array([[0.49923548, 0.49939131, 0.49992427, 0.49836011],
        [0.65380567, 0.57747782, 0.40608983, 0.59908658],
        [0.5598664 , 0.64054905, 0.55804003, 0.28263748],
        [0.70671607, 0.70942866, 0.4634057 , 0.37210521]]),
 array([[0.1418173 ],
        [0.19649449],
        [0.11108003],
        [0.16115001]]),
 array([0.53539502, 0.54896618, 0.52774149, 0.54020054]))

In [34]:
loss = -np.mean(y * np.log(y_pred + 1e-8) +
    (1 - y) * np.log(1 - y_pred + 1e-8))
loss

np.float64(0.7065989917203735)

In [37]:
for ep in range(1, 1000 + 1):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2).ravel()
    y_pred =  a2
    
    loss = -np.mean(y * np.log(y_pred + 1e-8) +
                    (1 - y) * np.log(1 - y_pred + 1e-8))
    
    if ep % 100 == 0 or ep == 1:
        acc = (y_pred.round() == y).mean()
        print(f"epoch {ep:5d}  loss={loss:.4f}  acc={acc:.2f}")
                
    N = len(X)
    y_pred = y_pred.reshape(-1, 1)

    dz2 = (y_pred - y.reshape(-1, 1))
    dW2 = (a1.T @ dz2) / N
    db2 = dz2.mean(axis=0)

    da1 = dz2 @ W2.T
    dz1 = da1 * dsigmoid(a1)
    dW1 = (X.T @ dz1) / N
    db1 = dz1.mean(axis=0)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2    

epoch     1  loss=0.6956  acc=0.50
epoch   100  loss=0.6922  acc=0.50
epoch   200  loss=0.6902  acc=0.50
epoch   300  loss=0.6755  acc=0.50
epoch   400  loss=0.5753  acc=0.75
epoch   500  loss=0.3638  acc=1.00
epoch   600  loss=0.1134  acc=1.00
epoch   700  loss=0.0524  acc=1.00
epoch   800  loss=0.0326  acc=1.00
epoch   900  loss=0.0233  acc=1.00
epoch  1000  loss=0.0180  acc=1.00
